<a href="https://colab.research.google.com/github/hatamimatt/ClimateDataScience/blob/main/nex_gddp_cmip6_extremeRainfallOrlandoSingleModelPipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# STEP 1 — Install packages
# ============================================================

!pip install xarray dask h5netcdf s3fs cftime pandas numpy matplotlib rioxarray rasterio -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.6/202.6 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.3/88.3 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 58.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.3.0 which is incompatible.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.3.0 which is incompatible.


In [ ]:
# ============================================================
# STEP 2 — Import libraries
# ============================================================

import s3fs
import xarray as xr
import rioxarray as rxr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# ============================================================
# STEP 3 — Connect to NEX-GDDP-CMIP6 on AWS
# ============================================================

fs = s3fs.S3FileSystem(anon=True)

fs.ls("nex-gddp-cmip6/NEX-GDDP-CMIP6")[:5]

In [ ]:
# ============================================================
# STEP 4 — Project settings
# ============================================================

# Orlando / Central Florida test bounding box
lat_min, lat_max = 28.0, 29.0
lon_min, lon_max = -82.0, -80.5

model = "ACCESS-CM2"
variant = "r1i1p1f1"
scenario = "ssp245"
variable = "pr"

baseline_years = range(1981, 2011)
future_years = range(2071, 2101)

In [ ]:
# ============================================================
# STEP 5 — Find one latest NEX-GDDP file per year
# ============================================================

def get_latest_files_by_year(model, scenario, variant, variable, years):
    folder = f"nex-gddp-cmip6/NEX-GDDP-CMIP6/{model}/{scenario}/{variant}/{variable}"

    all_files = fs.glob(folder + "/*.nc")
    selected_files = []

    for year in years:
        year_files = [f for f in all_files if f"_{year}" in f]

        if len(year_files) == 0:
            print(f"No file found for {model}, {scenario}, {year}")
            continue

        v12_files = [f for f in year_files if "_v1.2.nc" in f]
        v11_files = [f for f in year_files if "_v1.1.nc" in f]

        if len(v12_files) > 0:
            chosen_file = v12_files[0]
        elif len(v11_files) > 0:
            chosen_file = v11_files[0]
        else:
            chosen_file = year_files[0]

        selected_files.append("s3://" + chosen_file)

    return selected_files

In [ ]:
# ============================================================
# STEP 6 — Open precipitation and subset to Orlando area
# ============================================================

def open_precip_for_bbox(files):
    ds = xr.open_mfdataset(
        files,
        engine="h5netcdf",
        combine="by_coords",
        chunks={"time": 90},
        backend_kwargs={"storage_options": {"anon": True}}
    )

    pr = ds["pr"]

    # Convert kg m-2 s-1 to mm/day
    pr = pr * 86400
    pr.attrs["units"] = "mm/day"

    # Convert longitude from 0–360 to -180–180
    pr = pr.assign_coords(
        lon=((pr.lon + 180) % 360) - 180
    ).sortby("lon")

    # Subset bounding box
    pr_bbox = pr.sel(
        lat=slice(lat_min, lat_max),
        lon=slice(lon_min, lon_max)
    )

    return pr_bbox

In [ ]:
# ============================================================
# STEP 7 — Load historical and future precipitation
# ============================================================

hist_files = get_latest_files_by_year(
    model=model,
    scenario="historical",
    variant=variant,
    variable=variable,
    years=baseline_years
)

future_files = get_latest_files_by_year(
    model=model,
    scenario=scenario,
    variant=variant,
    variable=variable,
    years=future_years
)

print("Historical files:", len(hist_files))
print("Future files:", len(future_files))

hist_pr = open_precip_for_bbox(hist_files)
future_pr = open_precip_for_bbox(future_files)

hist_pr

In [ ]:
# ============================================================
# STEP 8 — Load Atlas 14 10-year 24-hour threshold
# ============================================================

atlas10 = rxr.open_rasterio("se10yr24ha.asc", masked=True)
atlas10 = atlas10.squeeze("band", drop=True)
atlas10 = atlas10.rename({"x": "lon", "y": "lat"})

# Atlas 14 ASCII values appear to be stored as thousandths of inches.
# Convert to mm:
# raw / 1000 = inches
# inches * 25.4 = mm
atlas10_mm = (atlas10 / 1000) * 25.4
atlas10_mm.attrs["units"] = "mm"

In [ ]:
# ============================================================
# STEP 9 — Interpolate Atlas 14 threshold to NEX-GDDP grid
# ============================================================

atlas10_on_grid = atlas10_mm.interp(
    lat=hist_pr.lat,
    lon=hist_pr.lon
)

print("Atlas 10-year 24-hour min mm:", float(atlas10_on_grid.min().compute()))
print("Atlas 10-year 24-hour max mm:", float(atlas10_on_grid.max().compute()))

plt.figure(figsize=(7, 5))
atlas10_on_grid.plot()
plt.title("Atlas 14 10-year 24-hour rainfall threshold: Orlando area")
plt.show()

In [ ]:
# ============================================================
# STEP 10 — Raw model exceedance test
# ============================================================

hist_exceed_raw = hist_pr > atlas10_on_grid
future_exceed_raw = future_pr > atlas10_on_grid

print("Raw historical exceedances:", int(hist_exceed_raw.sum().compute()))
print("Raw future exceedances:", int(future_exceed_raw.sum().compute()))

print("Historical max daily rainfall:", float(hist_pr.max().compute()))
print("Future max daily rainfall:", float(future_pr.max().compute()))

In [ ]:
# ============================================================
# STEP 11 — Simple bias correction using historical annual maxima
# ============================================================

# Annual maximum daily rainfall from historical model data
hist_annual_max = hist_pr.resample(time="YS").max()

# A 10-year-like threshold from 30 years of model data
# 0.90 quantile of annual maxima roughly corresponds to a high return-period event
model_hist_10yr_like = hist_annual_max.quantile(0.90, dim="time").compute()

# Correction factor aligns model historical extreme scale with Atlas 14 threshold
correction_factor = atlas10_on_grid / model_hist_10yr_like

# Avoid unrealistic correction values
correction_factor = correction_factor.clip(min=0.5, max=5)

print("Correction factor min:", float(correction_factor.min().compute()))
print("Correction factor max:", float(correction_factor.max().compute()))

plt.figure(figsize=(7, 5))
correction_factor.plot()
plt.title("Simple correction factor")
plt.show()

In [ ]:
# ============================================================
# STEP 12 — Apply correction and recalculate exceedance
# ============================================================

hist_pr_corrected = hist_pr * correction_factor
future_pr_corrected = future_pr * correction_factor

hist_exceed = hist_pr_corrected > atlas10_on_grid
future_exceed = future_pr_corrected > atlas10_on_grid

print("Corrected historical exceedances:", int(hist_exceed.sum().compute()))
print("Corrected future exceedances:", int(future_exceed.sum().compute()))

In [ ]:
# ============================================================
# STEP 13 — Annual exceedance frequency
# ============================================================

hist_annual_exceed = hist_exceed.resample(time="YS").sum()
future_annual_exceed = future_exceed.resample(time="YS").sum()

hist_mean_exceed = hist_annual_exceed.mean(dim="time")
future_mean_exceed = future_annual_exceed.mean(dim="time")

change_exceed = future_mean_exceed - hist_mean_exceed
ratio_exceed = future_mean_exceed / hist_mean_exceed.where(hist_mean_exceed > 0)

In [ ]:
# ============================================================
# STEP 14 — Plot results
# ============================================================

plt.figure(figsize=(7, 5))
hist_mean_exceed.plot()
plt.title("Historical annual exceedance frequency\n10-year 24-hour threshold")
plt.show()

plt.figure(figsize=(7, 5))
future_mean_exceed.plot()
plt.title("Future annual exceedance frequency\n10-year 24-hour threshold")
plt.show()

plt.figure(figsize=(7, 5))
change_exceed.plot(cmap="RdBu_r")
plt.title("Change in annual exceedance frequency\nFuture minus historical")
plt.show()

In [ ]:
# ============================================================
# STEP 15 — Simple summary table
# ============================================================

summary = {
    "model": model,
    "scenario": scenario,
    "threshold": "Atlas 14 10-year 24-hour",
    "historical_mean_exceedances_per_year": float(hist_mean_exceed.mean().compute()),
    "future_mean_exceedances_per_year": float(future_mean_exceed.mean().compute()),
    "change_exceedances_per_year": float(change_exceed.mean().compute())
}

summary_df = pd.DataFrame([summary])
summary_df